# Energy Consumption (Plymouth, LSOA 2024)

This notebook loads DESNZ domestic gas and electricity consumption data,
filters to Plymouth LSOAs, merges off-gas property counts and joins to spatial
boundaries.

## 1. Imports

In [ ]:
#Import necessary libraries
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


PROJECT_DIR = next(candidate
                   for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                   if (candidate / "01_Data").exists()
                   )


## 2. Configure Paths

In [ ]:
# configure path for data directories
RAW_ENERGY_DIR = PROJECT_DIR / "01_Data/Raw/DESNZ_Energy"
PROCESSED_BOUNDARY_DIR = PROJECT_DIR / "01_Data/Processed/Boundaries"
PROCESSED_ENERGY_DIR = PROJECT_DIR / "01_Data/Processed/Energy"
PROCESSED_ENERGY_DIR.mkdir(parents=True, exist_ok=True)
BOUNDARY_GPKG = PROCESSED_BOUNDARY_DIR / "plymouth_boundaries.gpkg"

In [ ]:
# Load Plymouth boundary layers — use clipped LSOA so geometries stay inside LAD boundary
lad_plymouth = gpd.read_file(BOUNDARY_GPKG, layer="lad_plymouth_2024")
plymouth_lsoa = gpd.read_file(BOUNDARY_GPKG, layer="lsoa_plymouth_2021_clipped")

print(len(lad_plymouth), lad_plymouth.crs)
print(len(plymouth_lsoa), plymouth_lsoa.crs)

## 3. Load Energy Data

In [ ]:
#load the domestic gas consumption data
gas_lsoa = pd.read_csv(RAW_ENERGY_DIR / "DESNZ_domestic_gas_LSOA_2010_2024.csv", encoding="latin-1")

## 4. EDA

In [ ]:
#check shape of the gas_lsoa dataframe
gas_lsoa.shape

In [ ]:

gas_lsoa.info()

In [ ]:
gas_lsoa.isnull().sum()

In [ ]:
print(gas_lsoa.columns.tolist())

In [ ]:
gas_lsoa.head()

In [ ]:
gas_lsoa.Year.unique()

In [ ]:
# Filter to Plymouth LA code
gas_lsoa = gas_lsoa[gas_lsoa["LA_code"] == "E06000026"]

In [ ]:
#check for nulls in the mean consumption column for plymouth lsoas
gas_lsoa.isnull().sum()

In [ ]:
# Filter to 2024 only
gas_lsoa = gas_lsoa[gas_lsoa["Year"] == 2024].copy()

In [ ]:
# Check LSOA code coverage between boundary and energy data
print(f"Boundary LSOAs: {plymouth_lsoa['LSOA21CD'].nunique()}")
print(f"Gas LSOAs: {gas_lsoa['LSOA_code'].nunique()}")

In [ ]:
#check if all energy LSOA codes are in the boundary LSOA codes
gas_lsoa[~gas_lsoa["LSOA_code"].isin(plymouth_lsoa["LSOA21CD"])]

In [ ]:
#rename  columns for clarity 
gas_lsoa_clean = gas_lsoa.rename(columns={
    "Meters": "gas_meters",
    "Consumption_kWh": "gas_consumption_kwh",
    "Mean_consumption_kWh_per_meter": "gas_mean_kwh_per_meter",
    "Median_consumption_kWh_per_meter": "gas_median_kwh_per_meter"
}).copy()

In [ ]:
print(gas_lsoa_clean.columns)

In [ ]:
# create a copy of the gas_lsoa_clean dataframe for further analysis
plymouth_lsoa_energy_consumption = gas_lsoa_clean.copy()

In [ ]:
plymouth_lsoa_energy_consumption.shape

## 5. Off-Gas Data

In [ ]:
#inspect off gas data
off_gas_path = RAW_ENERGY_DIR / "DESNZ_LSOA_properties_not_connected_to_gas_network_2015_2024.xlsx"

xls = pd.ExcelFile(off_gas_path)

print(xls.sheet_names)

In [ ]:
#load 2024 sheet
off_gas = pd.read_excel(off_gas_path, sheet_name="2024", skiprows=3)  

In [ ]:
#check shape of off gas data
off_gas.shape

In [ ]:
#filter to plymouth lsoas only
off_gas_plymouth_2024 = off_gas[off_gas["LSOA code"].isin(plymouth_lsoa["LSOA21CD"])].copy()

In [ ]:
off_gas_plymouth_2024.shape

In [ ]:
off_gas_plymouth_2024.columns.tolist()

In [ ]:
off_gas_plymouth_2024.head(20)

In [ ]:
off_gas_cols = ["LA_code", "LA", "MSOA_code", "MSOA", "LSOA_code", "LSOA",
                "domestic_properties", "domestic_gas_meters_offgas_file",
                "off_gas_properties", "off_gas_property_share"]

off_gas_plymouth_2024 = off_gas_plymouth_2024.rename(columns={
    "Local authority code": "LA_code",
    "Local authority": "LA",
    "MSOA code": "MSOA_code",
    "Middle layer super output area": "MSOA",
    "LSOA code": "LSOA_code",
    "Lower layer super output area": "LSOA",
    "Number of \ndomestic \nproperties": "domestic_properties",
    "Number of \ndomestic \ngas meters": "domestic_gas_meters_offgas_file",
    "Estimated number \nof properties not \non the gas grid": "off_gas_properties",
    "Estimated percentage\nof properties not \non the gas grid": "off_gas_property_share"
})[off_gas_cols].copy()

In [ ]:
off_gas_plymouth_2024.isnull().sum()

In [ ]:
#merge the off gas data with the plymouth_lsoa_energy_consumption dataframe
plymouth_lsoa_energy_consumption = plymouth_lsoa_energy_consumption.merge(
    off_gas_plymouth_2024[["LSOA_code","domestic_properties","domestic_gas_meters_offgas_file",
            "off_gas_properties","off_gas_property_share"]],
            on="LSOA_code",how="left",validate="one_to_one"
            )

In [ ]:
print(plymouth_lsoa_energy_consumption.shape)

#check for missing off-gas records after merging 
print("Missing off-gas records:", plymouth_lsoa_energy_consumption["off_gas_properties"].isna().sum())

In [ ]:
#Join energy consumption table to Plymouth LSOA spatial layer
plymouth_lsoa_energy_gdf = plymouth_lsoa.merge(
    plymouth_lsoa_energy_consumption,
    left_on="LSOA21CD",
    right_on="LSOA_code",
    how="left",
    validate="one_to_one"
)

## 6. Spatial Join

In [ ]:
#check final GDF for missing data
print(f"LSOA energy GDF: {plymouth_lsoa_energy_gdf.shape}")
print(f"Missing gas: {plymouth_lsoa_energy_gdf['gas_consumption_kwh'].isna().sum()}")
print(f"Missing off-gas: {plymouth_lsoa_energy_gdf['off_gas_properties'].isna().sum()}")

In [ ]:
#check crs of the final GDF
plymouth_lsoa_energy_gdf.crs

In [ ]:
# calculate area in km2 for each LSOA and derive energy and property density indicators

plymouth_lsoa_energy_gdf["area_km2"] = (
    plymouth_lsoa_energy_gdf.geometry.area / 1_000_000
)

plymouth_lsoa_energy_gdf["gas_consumption_mwh"] = (
    plymouth_lsoa_energy_gdf["gas_consumption_kwh"] / 1_000
)

plymouth_lsoa_energy_gdf["gas_consumption_mwh_per_km2"] = (
    plymouth_lsoa_energy_gdf["gas_consumption_mwh"] /
    plymouth_lsoa_energy_gdf["area_km2"]
)

plymouth_lsoa_energy_gdf["domestic_properties_per_km2"] = (
    plymouth_lsoa_energy_gdf["domestic_properties"] /
    plymouth_lsoa_energy_gdf["area_km2"]
)

plymouth_lsoa_energy_gdf["off_gas_properties_per_km2"] = (
    plymouth_lsoa_energy_gdf["off_gas_properties"] /
    plymouth_lsoa_energy_gdf["area_km2"]
)

In [ ]:
#plotting the domestic gas consumption density by LSOA in Plymouth
fig, ax = plt.subplots(figsize=(10, 5))

plymouth_lsoa_energy_gdf.plot(
    column="gas_consumption_mwh_per_km2",
    cmap="OrRd",
    legend=True,
    legend_kwds={"label": "Gas Consumption (MWh per km²)"},
    ax=ax,
    edgecolor="grey",
    linewidth=0.3
)
lad_plymouth.boundary.plot(ax=ax, edgecolor="black", linewidth=1.2)

ax.set_title("Domestic Gas Consumption Density by LSOA")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
#check summary statistics for the energy and property density indicators
plymouth_lsoa_energy_gdf[
    [
        "area_km2",
        "gas_consumption_mwh",
        "gas_consumption_mwh_per_km2",
        "domestic_properties_per_km2",
        "off_gas_properties",
        "off_gas_property_share",
        "off_gas_properties_per_km2"
    ]
].describe()

## Save Outputs

In [ ]:
#save outputs to processed energy directory
lsoa_energy_csv = PROCESSED_ENERGY_DIR / "plymouth_lsoa_energy_consumption_2024.csv"
energy_gpkg = PROCESSED_ENERGY_DIR / "plymouth_energy_consumption_2024.gpkg"
plymouth_lsoa_energy_consumption.to_csv(lsoa_energy_csv, index=False)
plymouth_lsoa_energy_gdf.to_file(energy_gpkg,layer="lsoa_energy_consumption_2024",driver="GPKG")